# 🧑‍💻 Session 6: Retrieval Augmented Generation (RAG)

RAG combines:
1. **Retriever** → Fetches relevant documents from a vector store.  
2. **LLM** → Generates context-aware responses using query + retrieved docs.  

In this session, we will:
- Store documents with **Gemini Embeddings**  
- Retrieve docs from **Chroma**  
- Use **Groq LLM** for answering questions  


In [5]:
# 📌 Install dependencies
!pip install -q langchain==1.0.5 langchain-community==0.4.1 langchain-groq==1.0.0 langchain-google-genai==3.0.1 chromadb==1.3.4 pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 6.7 MB/s eta 0:00:00


In [6]:
!pip show langchain langchain-community langchain-groq langchain-google-genai chromadb pypdf

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-google-genai
Vers

## 🔑 Setup API Keys
- Google Gemini API key → for embeddings  
- Groq API key → for LLM (LLaMA models)


In [3]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

## 📄 Step 1: Load Documents
We’ll load our scholarship info document


In [11]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/scholarship_info.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()

# Step 2: Split the document

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

splits = splitter.split_documents(doc)
print(f"✅ Split into {len(splits)} chunks.")

# Preview first chunk
print("\n--- First Chunk ---\n")
print(splits[0].page_content[:500])

✅ Split into 2 chunks.

--- First Chunk ---

Title: Scholarship Information 2025 
 
1. Eligibility: 
- Open to students in India pursuing undergraduate degrees. 
- Annual family income must be below ₹6,00,000. 
- Minimum 60% marks in the last qualifying exam. 
 
2. Documents Required: 
- Income certificate 
- Aadhaar card 
- Bank passbook 
- Marksheet 
 
3. Deadline: October 15, 2025 
 
4. Benefits: 
- ₹10,000 per semester for tuition 
- Book allowance of ₹3,000 per year 
 
5. How to Apply:


## 🗂️ Step 3: Store Documents with Gemini Embeddings in Chroma
We’ll use **GoogleGenerativeAIEmbeddings** for vector representation.


In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

# Create Chroma vector store
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="rag_chroma_db",
    collection_name="ipl_docs"
)

# Add documents
vector_store.add_documents(splits)

/tmp/ipython-input-1672483023.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


['69e63f53-7d49-40d9-9d76-a1e5f7d31c6f',
 '4a7e3d1d-3aa5-4de7-947c-abff9129069e']

## 🔎 Step 4: Create Retriever
Retriever fetches relevant chunks from Chroma.


In [14]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

## 🧠 Step 5: Initialize Groq LLM
We use Groq-hosted LLaMA 3.


In [16]:
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_groq import ChatGroq

# Load API key
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=200
)

## 🔗 Step 6: Create RAG Chain
Combine retriever + LLM into a RetrievalQA pipeline.


In [17]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

## 💬 Step 7: Ask Questions
Test RAG pipeline with cricket-related queries.


In [19]:
# Query 1
query = "Who is the eligible for the scholarship?"
response = qa_chain.invoke({"query": query})

print("Query:", query)
print("Answer:", response["result"])

# Query 2
query2 = "What the benefits of the scholarship?"
response2 = qa_chain.invoke({"query": query2})

print("\nQuery:", query2)
print("Answer:", response2["result"])


Query: Who is the eligible for the scholarship?
Answer: **Eligible candidates**

- **Nationality/Location:** Students residing in India.  
- **Course level:** Currently pursuing an undergraduate (bachelor’s) degree.  
- **Family income:** Annual family income must be **below ₹6,00,000**.  
- **Academic performance:** Minimum **60 % marks** in the last qualifying examination (e.g., 12th grade, diploma, etc.).  

Only students who meet all of these criteria are eligible to apply for the scholarship.

Query: What the benefits of the scholarship?
Answer: **Benefits of the scholarship**

- ₹10,000 per semester for tuition fees  
- ₹3,000 per year as a book allowance


# ✅ Summary
- Used **Google Gemini Embeddings** to vectorize documents  
- Stored + Retrieved docs from **Chroma**  
- Connected retriever with **Groq LLM**  
- Answered queries using **RAG pipeline**  

👉 Next session: **Advanced RAG – Custom Prompts, Comparisons, and Deep Dive**
